In [1]:
import numpy as np
from sklearn import datasets

#loading iris dataset
iris = datasets.load_iris()
X = iris["data"][:, (2,3)] #input data
y = (iris["target"] == 2).astype(int)
y = y.reshape([150,1]) #truth values

In [2]:
#activation functions
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
  s = sigmoid(z)
  return s * (1-s)

In [3]:
# MLP (Multi-Layer Perceptron) class definition
class MLP:
  """
    A Multi-Layer Perceptron (MLP) class for basic neural network tasks.
    Supports a single hidden layer with sigmoid activation.
  """
  def __init__(self, input_size, hidden_size, output_size, learning_rate = 0.01):
    """
      Initialize the MLP with architecture dimensions and learning rate.

      Args:
          input_size (int): Number of input features.
          hidden_size (int): Number of neurons in the hidden layer.
          output_size (int): Number of output neurons.
          learning_rate (float): Step size for weight updates.
    """
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size
    self.learning_rate = learning_rate

    # Randomly initialize weights for both layers (Input to Hidden, Hidden to Output)
    self.weights1 = np.random.randn(self.input_size, self.hidden_size)
    self.weights2 = np.random.randn(self.hidden_size, self.output_size)

    # Initialize bias terms to zero for both layers
    self.bias1 = np.zeros((1, hidden_size))
    self.bias2 = np.zeros((1, output_size))

  def fit(self, X, y, epochs = 1000):
    """
      Train the model using Forward Propagation and Backpropagation.

      Args:
          X (ndarray): Training features of shape (samples, input_size).
          y (ndarray): Target labels of shape (samples, output_size).
          epochs (int): Number of training iterations.
    """
    for epoch in range(epochs):
      # --- FEED FORWARD PHASE ---
      # Calculate weighted sum and apply activation for the hidden layer
      layer1 = X.dot(self.weights1)+self.bias1
      activation1 = sigmoid(layer1)

      # Calculate weighted sum and apply activation for the output layer
      layer2 = activation1.dot(self.weights2) + self.bias2
      activation2 = sigmoid(layer2)

      # --- BACKPROPAGATION PHASE ---
      # Calculate the difference between the prediction and the true label
      error = activation2 - y

      # Calculate gradient for the output layer weights and biases
      # Uses the chain rule: error * derivative of activation function
      d_weights2 = activation1.T.dot(error * sigmoid_derivative(layer2))
      d_bias2 = np.sum(error * sigmoid_derivative(layer2), axis=0, keepdims=True)

      # Calculate gradient for the hidden layer weights and biases
      # Propagates the output error back through the second set of weights
      error_hidden = error.dot(self.weights2.T) * sigmoid_derivative(layer1)
      d_weights1 = X.T.dot(error_hidden)
      d_bias1 = np.sum(error_hidden, axis=0, keepdims=True)

      # --- PARAMETER UPDATE (Gradient Descent) ---
      # Update weights and biases by subtracting a portion (learning rate) of the gradient
      self.weights2 -= self.learning_rate * d_weights2
      self.bias2 -= self.learning_rate * d_bias2
      self.weights1 -= self.learning_rate * d_weights1
      self.bias1 -= self.learning_rate * d_bias1

  def predict(self, X):
    """
      Pass input through the network and apply a 0.5 threshold.

      Returns:
          ndarray: Binary classification results (0 or 1).
    """
    # Pass input through the trained network (Feed Forward only)
    layer1 = X.dot(self.weights1) + self.bias1
    activation1 = sigmoid(layer1)
    layer2 = activation1.dot(self.weights2) + self.bias2
    activation2 = sigmoid(layer2)

    # Apply a 0.5 threshold to convert probabilities into binary classes (0 or 1)
    return (activation2 > 0.5).astype(int)

In [4]:
#training and using the model
mlp = MLP(input_size=2, hidden_size=4, output_size=1)

#training
mlp.fit(X, y)

#making predictions
y_pred = mlp.predict(X)

# evaluate the accuracy of the MLP
accuracy = np.mean(y_pred == y)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.95
